# Sample Scatter Plots

Translated from QMCPy's sample_scatter_plots.ipynb

Generates point sets from different discrete distributions and
true measures, printing summary statistics. For actual scatter plots,
run this data through your preferred Julia plotting library (Plots.jl,
Makie.jl, etc.).

This script outputs CSV-like data you can paste into a plotter.

In [ ]:
import Pkg

function activate_qmcju_project()
    for dir in (pwd(), normpath(joinpath(pwd(), "..")))
        if isfile(joinpath(dir, "Project.toml"))
            Pkg.activate(dir; io=devnull)
            return dir
        end
    end
    error("QMCJu.jl Project.toml not found. Start Jupyter in the repository root or demos/ directory.")
end

project_dir = activate_qmcju_project()
try
    @eval using Plots
    @eval using QMCJu
catch err
    if err isa ArgumentError
        Pkg.instantiate()
        @eval using Plots
        @eval using QMCJu
    else
        rethrow()
    end
end

using Statistics
using Printf

n = 128

IID Samples

In [ ]:
println("="^60)
println("IID Uniform Samples (n=$n, d=2)")
println("="^60)

dd_iid = IIDStdUniform(2; seed=7)
x_iid = gen_samples(dd_iid, n)
@printf("  Range: x₁ ∈ [%.4f, %.4f], x₂ ∈ [%.4f, %.4f]\n",
        minimum(x_iid[:,1]), maximum(x_iid[:,1]),
        minimum(x_iid[:,2]), maximum(x_iid[:,2]))
@printf("  Mean:  (%.4f, %.4f)\n", mean(x_iid[:,1]), mean(x_iid[:,2]))
println()

In [ ]:
# Scatter plot of IID samples
scatter(x_iid[:, 1], x_iid[:, 2],
    xlabel="x₁", ylabel="x₂",
    title="IID Uniform (n=$n)",
    label=nothing, markersize=2, alpha=0.7,
    aspect_ratio=:equal, xlims=(0,1), ylims=(0,1),
    size=(400, 400))

Low-Discrepancy Samples

In [ ]:
println("="^60)
println("Low-Discrepancy Samples (n=$n, d=2)")
println("="^60)

for (name, dd) in [
    ("Shifted Lattice",     Lattice(2; randomize=true, seed=7)),
    ("Scrambled Sobol'",    DigitalNetB2(2; randomize="LMS_DS", seed=7)),
    ("Generalized Halton",  Halton(2; randomize=true, seed=7)),
]
    local x
    x = gen_samples(dd, n)
    @printf("  %-20s: mean = (%.4f, %.4f), range x₁ = [%.4f, %.4f]\n",
            name, mean(x[:,1]), mean(x[:,2]),
            minimum(x[:,1]), maximum(x[:,1]))
end
println()

In [ ]:
# Scatter plots comparing low-discrepancy sequences
ld_plots = []
for (name, dd) in [
    ("Shifted Lattice",     Lattice(2; randomize=true, seed=7)),
    ("Scrambled Sobol'",    DigitalNetB2(2; randomize="LMS_DS", seed=7)),
    ("Generalized Halton",  Halton(2; randomize=true, seed=7)),
]
    local x = gen_samples(dd, n)
    p = scatter(x[:, 1], x[:, 2],
        xlabel="x₁", ylabel="x₂",
        title=name,
        label=nothing, markersize=2, alpha=0.7,
        aspect_ratio=:equal, xlims=(0,1), ylims=(0,1))
    push!(ld_plots, p)
end
plot(ld_plots..., layout=(1, 3), size=(900, 300))

Transformed Samples: Uniform → Gaussian

In [ ]:
println("="^60)
println("Transform to Gaussian Measure")
println("="^60)

for (name, dd) in [
    ("IID",     IIDStdUniform(2; seed=7)),
    ("Lattice", Lattice(2; randomize=true, seed=7)),
    ("Sobol'",  DigitalNetB2(2; seed=7)),
    ("Halton",  Halton(2; randomize=true, seed=7)),
]
    local tm, x, xt
    tm = Gaussian(dd; mean=0.0, covariance=1.0)
    x = gen_samples(dd, n)
    xt = transform(tm, x)
    @printf("  %-8s → N(0,1): mean = (%+.3f, %+.3f), std = (%.3f, %.3f)\n",
            name, mean(xt[:,1]), mean(xt[:,2]), std(xt[:,1]), std(xt[:,2]))
end
println()

In [ ]:
# Scatter plots: Uniform → Gaussian transform
gauss_plots = []
for (name, dd) in [
    ("IID",     IIDStdUniform(2; seed=7)),
    ("Lattice", Lattice(2; randomize=true, seed=7)),
    ("Sobol'",  DigitalNetB2(2; seed=7)),
    ("Halton",  Halton(2; randomize=true, seed=7)),
]
    local tm = Gaussian(dd; mean=0.0, covariance=1.0)
    local x = gen_samples(dd, n)
    local xt = transform(tm, x)
    p = scatter(xt[:, 1], xt[:, 2],
        xlabel="x₁", ylabel="x₂",
        title="$name → N(0,1)",
        label=nothing, markersize=2, alpha=0.7, color=:red,
        aspect_ratio=:equal, xlims=(-3,3), ylims=(-3,3))
    push!(gauss_plots, p)
end
plot(gauss_plots..., layout=(1, 4), size=(1000, 280))

Transformed Samples: Uniform → BrownianMotion

In [ ]:
println("="^60)
println("Transform to Brownian Motion")
println("="^60)

for (name, dd) in [
    ("IID",     IIDStdUniform(2; seed=7)),
    ("Lattice", Lattice(2; randomize=true, seed=7)),
    ("Sobol'",  DigitalNetB2(2; seed=7)),
    ("Halton",  Halton(2; randomize=true, seed=7)),
]
    local tm, x, xt
    tm = BrownianMotion(dd)
    x = gen_samples(dd, n)
    xt = transform(tm, x)
    @printf("  %-8s → BM: mean = (%+.3f, %+.3f), var = (%.3f, %.3f)\n",
            name, mean(xt[:,1]), mean(xt[:,2]), var(xt[:,1]), var(xt[:,2]))
end
println("  (BM: Var[W(t₁)] = t₁ = 0.5, Var[W(t₂)] = t₂ = 1.0)")
println()

In [ ]:
# Scatter plots: Uniform → Brownian Motion
bm_plots = []
for (name, dd) in [
    ("IID",     IIDStdUniform(2; seed=7)),
    ("Lattice", Lattice(2; randomize=true, seed=7)),
    ("Sobol'",  DigitalNetB2(2; seed=7)),
    ("Halton",  Halton(2; randomize=true, seed=7)),
]
    local tm = BrownianMotion(dd)
    local x = gen_samples(dd, n)
    local xt = transform(tm, x)
    p = scatter(xt[:, 1], xt[:, 2],
        xlabel="W(t₁)", ylabel="W(t₂)",
        title="$name → BM",
        label=nothing, markersize=2, alpha=0.7, color=:green,
        aspect_ratio=:equal, xlims=(-3,3), ylims=(-3,3))
    push!(bm_plots, p)
end
plot(bm_plots..., layout=(1, 4), size=(1000, 280))

Shifted & Stretched: Custom Uniform and Gaussian

In [ ]:
println("="^60)
println("Shifted & Stretched Measures (Sobol')")
println("="^60)

dd = DigitalNetB2(2; randomize="LMS_DS", seed=7)
x = gen_samples(dd, n)

Custom Uniform[2,4] × Uniform[6,8]

In [ ]:
tm_u = Uniform(dd; lower_bound=[2.0, 6.0], upper_bound=[4.0, 8.0])
xu = transform(tm_u, x)
@printf("  Uniform([2,4]×[6,8]): mean = (%.2f, %.2f)\n", mean(xu[:,1]), mean(xu[:,2]))

In [ ]:
# Scatter plot: Custom Uniform[2,4] × Uniform[6,8]
scatter(xu[:, 1], xu[:, 2],
    xlabel="x₁", ylabel="x₂",
    title="Sobol' → Uniform([2,4]×[6,8])",
    label=nothing, markersize=2, alpha=0.7, color=:blue,
    aspect_ratio=:equal, xlims=(1.5, 4.5), ylims=(5.5, 8.5),
    size=(400, 400))

Custom Gaussian with covariance

In [ ]:
Σ = [9.0 5.0; 5.0 9.0]
tm_g = Gaussian(dd; mean=[3.0, 7.0], covariance=Σ)
xg = transform(tm_g, gen_samples(DigitalNetB2(2; seed=7), n))
@printf("  Gaussian(μ=[3,7], Σ): mean = (%.2f, %.2f)\n", mean(xg[:,1]), mean(xg[:,2]))
println()

In [ ]:
# Scatter plot: Custom Gaussian with covariance
scatter(xg[:, 1], xg[:, 2],
    xlabel="x₁", ylabel="x₂",
    title="Sobol' → N([3,7], Σ)",
    label=nothing, markersize=2, alpha=0.7, color=:purple,
    size=(400, 400))

Keister Function Evaluation

In [ ]:
println("="^60)
println("Keister Function Integration (d=2)")
println("="^60)

dd = IIDStdUniform(2; seed=7)
tm = Gaussian(dd; covariance=0.5)
f = Keister(tm)
sc = CubMCCLT(f; abs_tol=0.3)
result = integrate(sc)
exact = keister_exact(2)
@printf("  Solution: %.4f (exact = %.4f)\n", result.solution, exact)
@printf("  Samples:  %d\n", result.data[:n])

println()
println("="^60)
println("Sample scatter plots demo completed!")
println("  (Use Plots.jl or Makie.jl to visualize the point sets)")